# 3-way (triplet) spatial colocalization

Derive a per-cell **3-way colocalization** number for marker triplets (A,B,C) from the pairwise z-scores already in `obsm['spatial_raw']` (pairs A/B, B/C, A/C).

**Metric (primary):** map each pair z to a bounded pseudo-correlation `rho=tanh(z/s)`, form the 3x3 correlation matrix `R`, and use `1 - det(R)` where `det(R)=1+2*rAB*rBC*rAC-(rAB^2+rBC^2+rAC^2)`.
- `-0.5*log det(R)` is the **Gaussian total correlation** (total dependence among the three).
- `det<0` (`1-det>1`) marks a **frustrated** triad: pairwise values not jointly realizable.
- `det` decomposes into a **bottleneck** (`min rho`) and a **balance / pure-3-way** term (`2*rAB*rBC*rAC`).

Caveat: a pair-derived number is a *proxy* for true 3-way co-occurrence — see the validation arm at the end.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')           # for `import PixelGen.*`
sys.path.append('/home/projects/nyosef/zvise/PixelGen/PixelGen')   # for pxl_utils' bare `scvi_utils`/`common_utils`
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import pyplot as plt

NEW_DATA = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
CACHE = NEW_DATA / 'cache'
sys.path.append(str(NEW_DATA)); import nalm_utils as nu
from PixelGen.utils import build_triplet_obsm, get_marker_triplets
sc.set_figure_params(figsize=(5, 3), frameon=False)

adata = sc.read_h5ad(CACHE / 'adata_cytovi_annotated_compat.h5ad')
print(adata.obsm['spatial_raw'].shape, '<- pairwise z-scores')

## Build triplet modalities
Determinant (primary) + its two interpretable components on the CD8 panel, plus a B-panel determinant. The cross-system and UMAP sections below all run on a **focused `tri_sel`** panel: the top-100 most-variable triplets in B cells + top-100 in CD8 cells (≤200 total) — keeps the differential plots legible.

In [ ]:
pair_markers = {m for col in adata.obsm['spatial_raw'].columns for m in col.split('/')}
panels = json.load(open(NEW_DATA / 'marker_panels.json'))
panel = panels['cd8_t_cell_markers']
panel = [m for grp in panel.values() for m in grp] if isinstance(panel, dict) else panel
markers_tri = list(dict.fromkeys(m for m in panel if m in pair_markers))
triplets = get_marker_triplets(markers_tri, adata.obsm['spatial_raw'].columns)
print(len(markers_tri), 'CD8-panel markers present ->', len(triplets), 'valid triplets')

# raw values so we can inspect frustration (det<0) and the decomposition
for metric, key in [('determinant','tri_det'), ('triple_product','tri_prod'), ('min','tri_min')]:
    build_triplet_obsm(adata, source_key='spatial_raw', markers=markers_tri,
                       metric=metric, value_transform='raw', target_key=key)

# B-cell determinant modality (B panel: identity + Ig + APC + inhibitory + Breg + MHC)
b_markers = [m for grp in panels['or'].values() for m in grp] + ['HLA-DR', 'HLA-ABC', 'HLA-DQ', 'B2M']
b_markers = [m for m in dict.fromkeys(b_markers) if m in pair_markers]
build_triplet_obsm(adata, source_key='spatial_raw', markers=b_markers,
                   metric='determinant', value_transform='raw', target_key='tri_det_b')

# Focused panel: top-100 most-variable triplets within B cells (B panel) + within CD8 cells (CD8 panel)
ct = adata.obs['cell_type_annot']
b_top = adata.obsm['tri_det_b'].loc[ct == 'B'].var().nlargest(100).index
t_top = adata.obsm['tri_det'].loc[ct == 'CD8'].var().nlargest(100).index
tri_sel = pd.concat([adata.obsm['tri_det_b'][b_top], adata.obsm['tri_det'][t_top]], axis=1)
adata.obsm['tri_sel'] = tri_sel.loc[:, ~tri_sel.columns.duplicated()]
print(f"focused panel 'tri_sel': {adata.obsm['tri_sel'].shape[1]} triplets (top 100 B + top 100 CD8 by variance)")

## How many triads are *frustrated*?
`1-det>1` means the 3x3 pseudo-correlation matrix is not PSD: the three pairwise values cannot co-exist in one joint distribution.

In [ ]:
det = adata.obsm['tri_det'].values.ravel()
frac = (det > 1).mean()
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(det, bins=80, color='slateblue')
ax.axvline(1, color='k', ls='--', lw=1)
ax.set(xlabel='1 - det(R)   ( >1 = frustrated )', ylabel='cell-triplets')
ax.set_title(f'{frac:.1%} of cell-triplets are frustrated (det<0)')
plt.tight_layout(); plt.show()

## Determinant = bottleneck + balance
The determinant is high when *all three* pairs are strong (bottleneck, `min rho`) **and** the triad is balanced (`triple product`). Plotting against each component shows what drives a given triad.

In [ ]:
rng = np.random.default_rng(0)
d = adata.obsm['tri_det'].values.ravel()
mn = adata.obsm['tri_min'].values.ravel()
pr = adata.obsm['tri_prod'].values.ravel()
idx = rng.choice(d.size, size=min(20000, d.size), replace=False)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].scatter(mn[idx], d[idx], s=2, alpha=0.2); axes[0].set(xlabel='min rho (bottleneck)', ylabel='1 - det')
axes[1].scatter(pr[idx], d[idx], s=2, alpha=0.2); axes[1].set(xlabel='triple product (balance)', ylabel='1 - det')
fig.suptitle('Determinant decomposes into bottleneck + balance')
plt.tight_layout(); plt.show()

## Differential triads between cell systems
Mann-Whitney U + BH-FDR on every triplet (`nalm_utils.compute_triplet_diff`).

In [ ]:
GROUP = 'cell_system'
A, B = 'NALM-6 + healthy T', 'healthy B + healthy T'
g = adata.obs[GROUP]
det_df = adata.obsm['tri_sel']
res = nu.compute_triplet_diff(det_df.loc[g == A], det_df.loc[g == B])

# rank significant hits by EFFECT SIZE.
res = res[res.padj < 0.05].reindex(
    res.loc[res.padj < 0.05, 'mean_diff'].abs().sort_values(ascending=False).index
)
print(f'{len(res)} triplets FDR<0.05; ranked by |mean_diff|   ({A}  vs  {B})')
res.head(15)

# Cross-system triplet comparison — HT/NALM vs HT/HB

Mirror of `ht_nalm_vs_ht_hb_analysis.ipynb` (shared healthy-T donor; B target = NALM-6 line vs primary healthy B), on the **3-way determinant** modality instead of pairwise coloc. All plots here run on the focused **`tri_sel`** panel (top-100 most-variable triplets in B cells + top-100 in CD8 cells). Split into **CD8** and **B**; each compartment gets:

1. **Cross-system diff** (NALM vs HB) — Mann-Whitney + BH-FDR, ranked by effect size (`|mean_diff|`); top triads gained/lost in NALM-6.
2. **Per-system Blina-vs-Mock MA** — mean determinant vs LFC, one panel per system (6h).
3. **Blina-response scatter** — per triplet, x = LFC(Blina−Mock) in HT/HB vs y = LFC(Blina−Mock) in HT/NALM. On-diagonal = same response in both systems; off-diagonal = system-specific.

Then the most **significant** (cross-system) and most **variable** triplets are colored on the UMAP — whole dataset and a UMAP recomputed on the two-system subset. (`res` from the pooled diff above is reused for significance ranking.)

In [ ]:
# --- Inline triplet helpers (reuse nalm_utils) ---
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from nalm_utils import split_triplet, display_name

SYS_HT_NALM = 'NALM-6 + healthy T'
SYS_HT_HB   = 'healthy B + healthy T'
OBSM_TRI    = 'tri_sel'   # focused 200-triplet panel (top-100 B + top-100 CD8 by variance)

def _tri_label(t):
    return ' / '.join(display_name(m) for m in split_triplet(t))

def triplet_ma_table(values_a, values_b, names):
    """Per-triplet MA table: mean_signal (a∪b), mean_a/b, lfc=a-b, MWU pval, BH padj."""
    rows = []
    for j, name in enumerate(names):
        a = np.asarray(values_a[:, j], float); b = np.asarray(values_b[:, j], float)
        if a.std() == 0 and b.std() == 0:
            pval = 1.0
        else:
            try: _, pval = mannwhitneyu(a, b, alternative='two-sided')
            except ValueError: pval = 1.0
        rows.append({'triplet': name, 'mean_signal': np.concatenate([a, b]).mean(),
                     'mean_a': a.mean(), 'mean_b': b.mean(),
                     'lfc': a.mean() - b.mean(), 'pval': pval})
    df = pd.DataFrame(rows)
    _, df['padj'], _, _ = multipletests(df['pval'].fillna(1.0), method='fdr_bh')
    return df

def cross_system_triplet_diff(adata, cell_type, obsm_key=OBSM_TRI):
    """compute_triplet_diff(NALM vs HB) on one cell_type; FDR<0.05, ranked by |mean_diff|."""
    m = ((adata.obs['cell_type_annot'] == cell_type) &
         adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB]))
    sub = adata[m]; g = sub.obs['cell_system']; tri = sub.obsm[obsm_key]
    res = nu.compute_triplet_diff(tri.loc[g == SYS_HT_NALM], tri.loc[g == SYS_HT_HB])
    res = res[res.padj < 0.05]
    return res.reindex(res['mean_diff'].abs().sort_values(ascending=False).index)

def plot_triplet_diff_bars(res, cell_type, n=12):
    """Top-n gained (red) and top-n lost (blue) triads, readable even when one side is empty."""
    pos = res[res['mean_diff'] > 0].nlargest(n, 'mean_diff')
    neg = res[res['mean_diff'] < 0].nsmallest(n, 'mean_diff')
    top = pd.concat([neg, pos]).sort_values('mean_diff')
    n_up, n_dn = int((res['mean_diff'] > 0).sum()), int((res['mean_diff'] < 0).sum())
    fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(top) + 1.2))
    ax.barh([_tri_label(t) for t in top['triplet']], top['mean_diff'],
            color=np.where(top['mean_diff'] > 0, 'firebrick', 'steelblue'))
    ax.axvline(0, color='k', lw=0.8)
    ax.margins(y=0.01); ax.tick_params(axis='y', labelsize=8)
    ax.set_xlabel('mean 3-way coloc diff  (HT/NALM − HT/HB)')
    ax.set_title(f'{cell_type}: triads gained (red) / lost (blue) in NALM-6'
                 f'   [{n_up} up, {n_dn} down of {len(res)} FDR<0.05]')
    plt.tight_layout(); plt.show()

def plot_triplet_ma(adata, systems, time_val, cell_type, obsm_key=OBSM_TRI,
                    fdr=0.05, top_label=10):
    """Per-system Blina−Mock MA panels on triplets. Returns {label:{'triplet':df,...}}."""
    def _subset(sys_val):
        m = ((adata.obs['time'] == time_val) &
             (adata.obs['cell_type_annot'] == cell_type) &
             (adata.obs['cell_system'] == sys_val))
        sub = adata[m]; cond = sub.obs['condition'].values; tri = sub.obsm[obsm_key]
        df = triplet_ma_table(tri.values[cond == 'Blinatumomab'],
                              tri.values[cond == 'Mock'], list(tri.columns))
        return df, int((cond == 'Blinatumomab').sum()), int((cond == 'Mock').sum())

    def _panel(ax, df, title):
        df = df.copy(); df['sig'] = df['padj'] < fdr
        ns = df[~df['sig']]; up = df[df['sig'] & (df['lfc'] > 0)]; dn = df[df['sig'] & (df['lfc'] < 0)]
        ax.scatter(ns['mean_signal'], ns['lfc'], s=10, color='lightgrey', alpha=.5, ec='none', label=f'n.s. ({len(ns)})')
        ax.scatter(dn['mean_signal'], dn['lfc'], s=14, color='#1f77b4', alpha=.8, ec='none', label=f'↑ Mock ({len(dn)})')
        ax.scatter(up['mean_signal'], up['lfc'], s=14, color='#d62728', alpha=.8, ec='none', label=f'↑ Blina ({len(up)})')
        ax.axhline(0, color='k', lw=.7, ls='--')
        sig = df[df['sig']].copy(); sig['abslfc'] = sig['lfc'].abs()
        for _, r in sig.nlargest(top_label, 'abslfc').iterrows():
            ax.annotate(_tri_label(r['triplet']), (r['mean_signal'], r['lfc']),
                        fontsize=6, xytext=(3, 2), textcoords='offset points')
        ax.set(xlabel='mean tri_det (Blina ∪ Mock)', ylabel='LFC (Blina − Mock)', title=title)
        ax.legend(fontsize=7, frameon=False)

    out = {}; n = len(systems)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4.5), squeeze=False)
    for c, (lab, sv) in enumerate(systems):
        df, nb, nm = _subset(sv); out[lab] = {'triplet': df, 'n_blina': nb, 'n_mock': nm}
        _panel(axes[0, c], df, f'{lab} · {cell_type} · {time_val}  (Blina {nb}/Mock {nm})')
    fig.suptitle(f'{cell_type} triplets — Blina vs Mock MA', y=1.02)
    plt.tight_layout(); plt.show()
    return out

def plot_triplet_lfc_scatter(ma_results, label_x, label_y, cell_type, time_val,
                             fdr=0.05, top_label=12, ax=None):
    """Per-triplet Blina−Mock response: x = LFC(label_x), y = LFC(label_y).

    Diagonal (y=x) = triads that respond equally in both systems; off-diagonal =
    system-specific Blinatumomab response. Red = significant in both systems.
    """
    x = ma_results[label_x]['triplet']; y = ma_results[label_y]['triplet']
    m = x[['triplet', 'lfc', 'padj']].merge(
        y[['triplet', 'lfc', 'padj']], on='triplet', suffixes=('_x', '_y'))
    m['either'] = (m['padj_x'] < fdr) | (m['padj_y'] < fdr)
    m['both']   = (m['padj_x'] < fdr) & (m['padj_y'] < fdr)
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ns = m[~m['either']]; one = m[m['either'] & ~m['both']]; bo = m[m['both']]
    ax.scatter(ns['lfc_x'], ns['lfc_y'], s=8, color='lightgrey', alpha=.4, ec='none', label=f'n.s. ({len(ns)})')
    ax.scatter(one['lfc_x'], one['lfc_y'], s=18, color='#9467bd', alpha=.5, ec='none', label=f'sig 1 side ({len(one)})')
    ax.scatter(bo['lfc_x'], bo['lfc_y'], s=30, color='#d62728', alpha=.9, ec='black', lw=.4, label=f'sig both ({len(bo)})')
    lim = float(np.nanmax(np.abs(np.r_[m['lfc_x'].values, m['lfc_y'].values]))) * 1.05 or 1.0
    ax.plot([-lim, lim], [-lim, lim], color='k', lw=.7, ls='--', zorder=0)
    ax.axhline(0, color='grey', lw=.5); ax.axvline(0, color='grey', lw=.5)
    m['div'] = (m['lfc_y'] - m['lfc_x']).abs()
    for _, r in m[m['either']].nlargest(top_label, 'div').iterrows():
        ax.annotate(_tri_label(r['triplet']), (r['lfc_x'], r['lfc_y']),
                    fontsize=6, xytext=(3, 2), textcoords='offset points')
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim),
           xlabel=f'LFC Blina−Mock   ({label_x})',
           ylabel=f'LFC Blina−Mock   ({label_y})',
           title=f'{cell_type} — Blina response: {label_y} (y) vs {label_x} (x), {time_val}')
    ax.set_aspect('equal', 'box'); ax.legend(fontsize=7, frameon=False)
    if own: plt.tight_layout(); plt.show()
    return m

def plot_triplet_delta_lfc(ma_results, label_a, label_b, cell_type, time_val,
                           fdr=0.05, top_label=12, print_top=15):
    """ΔLFC = LFC(label_a) − LFC(label_b) vs mean signal. Purple ↑a, green ↑b."""
    a, b = ma_results[label_a]['triplet'], ma_results[label_b]['triplet']
    m = a[['triplet', 'mean_signal', 'lfc', 'padj']].merge(
        b[['triplet', 'mean_signal', 'lfc', 'padj']], on='triplet', suffixes=('_a', '_b'))
    m['mean_signal_combined'] = (m['mean_signal_a'] + m['mean_signal_b']) / 2
    m['delta_lfc'] = m['lfc_a'] - m['lfc_b']
    m['either'] = (m['padj_a'] < fdr) | (m['padj_b'] < fdr)
    m['both']   = (m['padj_a'] < fdr) & (m['padj_b'] < fdr)
    m['color']  = np.where(m['delta_lfc'] >= 0, '#9467bd', '#2ca02c')
    fig, ax = plt.subplots(figsize=(7, 6))
    ns = m[~m['either']]; one = m[m['either'] & ~m['both']]; bo = m[m['both']]
    ax.scatter(ns['mean_signal_combined'], ns['delta_lfc'], s=8, color='lightgrey', alpha=.4, ec='none', label=f'n.s. ({len(ns)})')
    ax.scatter(one['mean_signal_combined'], one['delta_lfc'], s=16, c=one['color'], alpha=.5, ec='none', label=f'sig 1 side ({len(one)})')
    ax.scatter(bo['mean_signal_combined'], bo['delta_lfc'], s=28, c=bo['color'], alpha=.9, ec='black', lw=.4, label=f'sig both ({len(bo)})')
    ax.axhline(0, color='k', lw=.7, ls='--')
    m['absd'] = m['delta_lfc'].abs()
    for _, r in m[m['either']].nlargest(top_label, 'absd').iterrows():
        ax.annotate(_tri_label(r['triplet']), (r['mean_signal_combined'], r['delta_lfc']),
                    fontsize=6, color=r['color'], xytext=(3, 2), textcoords='offset points')
    ax.set(xlabel='mean tri_det (across systems)', ylabel=f'ΔLFC  ({label_a} − {label_b})',
           title=f'{cell_type} triplets — ΔLFC (Blina−Mock): {label_a} vs {label_b}, {time_val}')
    ax.legend(fontsize=7, frameon=False)
    plt.tight_layout(); plt.show()
    top = m.nlargest(print_top, 'absd')[['triplet', 'lfc_a', 'lfc_b', 'delta_lfc', 'padj_a', 'padj_b']].copy()
    top['triplet'] = top['triplet'].map(_tri_label)
    print(f'Top {print_top} |ΔLFC| triplets ({label_a} − {label_b}, {cell_type} {time_val}):')
    return top.round(3)

### CD8 cells — HT/NALM vs HT/HB
Same healthy-T donor in both systems, so differences isolate how the T cell responds to the NALM-6 line vs primary healthy B.

In [ ]:
# CD8: cross-system triplet diff (NALM vs HB)
res_cd8 = cross_system_triplet_diff(adata, 'CD8')
print(f'{len(res_cd8)} triplets FDR<0.05 (CD8, HT/NALM vs HT/HB), ranked by |mean_diff|')
plot_triplet_diff_bars(res_cd8, 'CD8')
res_cd8.head(15).assign(triplet=lambda d: d.triplet.map(_tri_label))

In [ ]:
# CD8: per-system Blina-vs-Mock MA + LFC scatter (HT/HB on x, HT/NALM on y), 6h
ma_cd8 = plot_triplet_ma(adata, [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)], '6h', 'CD8')
plot_triplet_lfc_scatter(ma_cd8, 'HT/HB', 'HT/NALM', cell_type='CD8', time_val='6h')

### B cells — HT/NALM vs HT/HB
The B compartment differs across systems (NALM-6 cell line vs primary healthy B); differences reflect the model-vs-primary gap, not a donor effect. Run on the focused `tri_sel` panel (x = LFC in HT/HB, y = LFC in HT/NALM); off-diagonal triads = system-specific Blinatumomab response.

In [ ]:
# B: cross-system triplet diff (NALM vs HB)
res_b = cross_system_triplet_diff(adata, 'B')
print(f'{len(res_b)} triplets FDR<0.05 (B, HT/NALM vs HT/HB), ranked by |mean_diff|')
plot_triplet_diff_bars(res_b, 'B')
res_b.head(15).assign(triplet=lambda d: d.triplet.map(_tri_label))

In [ ]:
# B: per-system Blina-vs-Mock MA + LFC scatter (HT/HB on x, HT/NALM on y), 6h
ma_b = plot_triplet_ma(adata, [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)], '6h', 'B')
plot_triplet_lfc_scatter(ma_b, 'HT/HB', 'HT/NALM', cell_type='B', time_val='6h')

## Top triplets on the UMAP
Most **significant** (pooled cross-system effect size, from `res` above) and most **variable** triplets, colored by per-cell `tri_det` — first on the whole dataset, then on the two-system subset.

In [ ]:
# Top significant (pooled cross-system effect size, from `res`) + top variable triplets
sig6 = res['triplet'].head(6).tolist()
var6 = adata.obsm[OBSM_TRI].var().nlargest(6).index.tolist()
sel = list(dict.fromkeys(sig6 + var6))
for t in sel:                                   # temp obs cols for sc.pl.umap coloring
    adata.obs[t] = adata.obsm[OBSM_TRI][t].values
print('significant:', [_tri_label(t) for t in sig6])
print('variable   :', [_tri_label(t) for t in var6])

# whole dataset — larger panels
sc.set_figure_params(figsize=(6, 5))
sc.pl.umap(adata, color=['cell_system', 'cell_type_annot'], ncols=2, frameon=False)
sc.pl.umap(adata, color=sig6, ncols=3, title=[_tri_label(t) for t in sig6],
           cmap='viridis', frameon=False)
sc.pl.umap(adata, color=var6, ncols=3, title=[_tri_label(t) for t in var6],
           cmap='magma', frameon=False)
sc.set_figure_params(figsize=(5, 3))

In [ ]:
# UMAP recomputed on the two-system subset only (NALM-6+hT and healthy B+hT)
adata2 = adata[adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB])].copy()
print(f'two-system subset: {adata2.n_obs} cells — recomputing neighbors + UMAP on X_CytoVI')
sc.pp.neighbors(adata2, use_rep='X_CytoVI')
sc.tl.umap(adata2)
for t in sel:                                   # per-cell tri_det for coloring
    adata2.obs[t] = adata2.obsm[OBSM_TRI][t].values

sc.set_figure_params(figsize=(6, 5))
sc.pl.umap(adata2, color=['cell_system', 'cell_type_annot'], ncols=2, frameon=False)
sc.pl.umap(adata2, color=sig6, ncols=3, title=[_tri_label(t) for t in sig6],
           cmap='viridis', frameon=False)
sc.pl.umap(adata2, color=var6, ncols=3, title=[_tri_label(t) for t in var6],
           cmap='magma', frameon=False)
sc.set_figure_params(figsize=(5, 3))

# remove the temporary obs columns
adata.obs.drop(columns=sel, inplace=True)